# 09. External procedural analysis — best proxy → ElasticNet → XGBoost

This notebook mirrors the **compact procedural plotting style of notebook 4 (`procedural_ccle`)** while avoiding repeated metric-specific code.

For each protein and external proteomics platform, we evaluate the three frozen same-gene proxy formulations — **raw RNA**, **CCLE-train z-score mapped to protein scale**, and **CCLE-train univariate linear regression** — and retain the best proxy for the metric. No proxy coefficient is refit on external protein values.

The analysis is then split by metric:

- **Pearson correlation:** best proxy → ElasticNet → XGBoost → unresolved at thresholds 0.30, 0.50, and 0.70.
- **Cosine similarity:** the same sequential cascade at thresholds 0.30, 0.50, and 0.70.
- **MSE:** **no hard success thresholds are used**. Instead, MSE is summarized by distribution quantiles (10th, 25th, 50th, 75th, and 90th percentiles) for the best proxy, ElasticNet, and XGBoost within each platform.

The best-of-three proxy choice is an **external oracle / upper-bound proxy comparison**, because the winning proxy is selected using the external evaluation metric itself. These outputs therefore remain separate from the prespecified single-linear-proxy primary validation analysis.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "9_external_cascade"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import warnings

# Keep the procedural pass bounded and reproducible.
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_thread_var, "1")

search_roots = []
if os.environ.get("PAPER_VALIDATION_MODULE_DIR"):
    search_roots.append(Path(os.environ["PAPER_VALIDATION_MODULE_DIR"]).expanduser().resolve())
cwd = Path.cwd().resolve()
for base in [cwd, *cwd.parents]:
    for candidate in [base, base / "paper_ready_external_validation", base / "notebooks", base / "notebooks2"]:
        if candidate not in search_roots:
            search_roots.append(candidate)
for candidate in search_roots:
    if (candidate / "paper_validation.py").exists():
        sys.path.insert(0, str(candidate))
        MODULE_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "paper_validation.py was not found. Keep the shared module beside the notebook or set "
        "PAPER_VALIDATION_MODULE_DIR."
    )

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import paper_validation as pv

PROJECT_ROOT = pv.detect_project_root()
RUN_ROOT = Path(os.environ.get(
    "PAPER_VALIDATION_RUN_ROOT",
    PROJECT_ROOT / "results_external_validation_paper_v1",
)).expanduser().resolve()
LAYOUT = pv.make_output_layout(RUN_ROOT)
MANIFEST_PATH = RUN_ROOT / "validation_manifest.json"

PLATFORM_ORDER = ["TMT", "iTRAQ", "LabelFree"]
PROXY_ORDER = ["RawProxy", "ZScoreProxy", "LinearProxy"]
MODEL_ORDER = ["BestProxy", "ElasticNet", "XGBoost"]
STAGE_ORDER = ["Best proxy", "ElasticNet", "XGBoost", "Unresolved"]
ALL_METRICS = ("pearson", "cosine", "mse")

PEARSON_THRESHOLDS = tuple(float(x) for x in os.environ.get(
    "PAPER_VALIDATION_PEARSON_THRESHOLDS", "0.30,0.50,0.70"
).split(","))
COSINE_THRESHOLDS = tuple(float(x) for x in os.environ.get(
    "PAPER_VALIDATION_COSINE_THRESHOLDS", "0.30,0.50,0.70"
).split(","))
MSE_QUANTILES = tuple(float(x) for x in os.environ.get(
    "PAPER_VALIDATION_MSE_QUANTILES", "0.10,0.25,0.50,0.75,0.90"
).split(","))
if MSE_QUANTILES != tuple(sorted(MSE_QUANTILES)) or any((q <= 0 or q >= 1) for q in MSE_QUANTILES):
    raise ValueError("MSE quantiles must be unique ascending values strictly between 0 and 1.")
if 0.50 not in MSE_QUANTILES:
    raise ValueError("MSE_QUANTILES must include 0.50 so the median is always reported.")

CASCADE_SPECS = {
    "pearson": {"label": "Pearson correlation", "thresholds": PEARSON_THRESHOLDS},
    "cosine": {"label": "Cosine similarity", "thresholds": COSINE_THRESHOLDS},
}
METRIC_DIRECTION = {"pearson": "higher", "cosine": "higher", "mse": "lower"}

print("Notebook 09 — best-of-three proxy procedural analysis")
print("Shared module:", MODULE_DIR / "paper_validation.py")
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Pearson thresholds:", PEARSON_THRESHOLDS)
print("Cosine thresholds:", COSINE_THRESHOLDS)
print("MSE analysis: quantiles only", MSE_QUANTILES)

## 1. Load the current validation contract and resolve the frozen notebook-7/8 result run

Notebook 09 uses two deliberately separate provenance concepts:

- **Current manifest** — used only to locate and checksum the current CCLE reference, proxy-parameter table, and cohort contract files.
- **Frozen result hash** — taken from notebook 8's `external_primary_metrics_compact` and cross-checked against notebook 7's completion record. This is the hash attached to every notebook-09 result.

A current/frozen manifest mismatch is allowed by default because notebook 6 also hashes normalization-QC figures, so a figure-only rerun can change the full manifest hash without changing the scientific evaluation contract. Set `PAPER_VALIDATION_STRICT_MANIFEST_MATCH=1` to require exact equality.

Because notebook 09 rebuilds Raw/Z-score/Linear proxies from the current contract files, section 2 performs an additional numerical compatibility audit: the rebuilt LinearProxy must reproduce the frozen notebook-7 LinearProxy metrics. If it does not, execution stops rather than mixing incompatible scientific runs.


In [ ]:
manifest = pv.load_manifest(MANIFEST_PATH)
CURRENT_MANIFEST_HASH = str(manifest["manifest_hash"])
STRICT_MANIFEST_MATCH = os.environ.get("PAPER_VALIDATION_STRICT_MANIFEST_MATCH", "0") == "1"

if "files" not in manifest or "cohort_contracts" not in manifest["files"]:
    raise RuntimeError(
        "Notebook 09 requires the full notebook-6 validation manifest with cohort contracts."
    )


def _verified_manifest_file(spec: dict, label: str) -> Path:
    """Resolve one current-manifest file and verify its recorded SHA-256."""
    path = Path(spec["path"])
    if not path.exists():
        raise FileNotFoundError(f"Current manifest file is missing: {label} -> {path}")
    expected_sha = str(spec.get("sha256", ""))
    if expected_sha:
        observed_sha = pv.file_sha256(path)
        if observed_sha != expected_sha:
            raise RuntimeError(
                f"Current manifest file checksum mismatch for {label}: "
                f"{observed_sha} != {expected_sha}"
            )
    return path


# Current notebook-6 contract files.  Their internal checksums must be valid even
# when the overall current manifest hash differs from the frozen evaluation hash.
reference_path = _verified_manifest_file(
    manifest["files"]["ccle_reference"], "ccle_reference"
)
proxy_parameter_path = _verified_manifest_file(
    manifest["files"]["linear_proxy_parameters"], "linear_proxy_parameters"
)

cohort_contracts = {}
for cohort, info in manifest["files"]["cohort_contracts"].items():
    cohort = str(cohort)
    cohort_contracts[cohort] = _verified_manifest_file(info, f"cohort_contract/{cohort}")

expected_cohorts = pv.COHORT_TABLE["cohort"].astype(str).tolist()
if set(cohort_contracts) != set(expected_cohorts):
    raise RuntimeError(
        f"Cohort-contract mismatch: expected {len(expected_cohorts)}, "
        f"found {len(cohort_contracts)}."
    )

reference = joblib.load(reference_path)
proxy_parameters = pv.read_table(proxy_parameter_path)

# -----------------------------------------------------------------------------
# Resolve the authoritative frozen result hash from notebook 8.
# -----------------------------------------------------------------------------
primary_metrics_path = LAYOUT["tables"] / "external_primary_metrics_compact"
primary_metrics = pv.read_table(primary_metrics_path)

if "manifest_hash" not in primary_metrics.columns:
    raise RuntimeError(
        "external_primary_metrics_compact has no manifest_hash column; "
        "rerun notebook 8 from the completed notebook-7 outputs."
    )

result_hashes = sorted(
    primary_metrics["manifest_hash"].dropna().astype(str).unique().tolist()
)
if len(result_hashes) != 1:
    raise RuntimeError(
        "external_primary_metrics_compact must contain exactly one frozen result hash; "
        f"found {result_hashes}."
    )

RESULT_MANIFEST_HASH = result_hashes[0]
MANIFEST_HASH = RESULT_MANIFEST_HASH
pv.assert_manifest_column(primary_metrics, MANIFEST_HASH, "external_primary_metrics_compact")

# Notebook 7's explicit completion record must agree with notebook 8 whenever it
# is available.  This catches accidental mixing of two completed evaluation runs.
completion_csv = LAYOUT["evaluation"] / "evaluation_completion.csv"
NOTEBOOK7_COMPLETION_VERIFIED = False
if completion_csv.exists() and completion_csv.stat().st_size > 0:
    completion_record = pd.read_csv(completion_csv, low_memory=False)
    if "manifest_hash" not in completion_record.columns:
        raise RuntimeError(
            "Notebook-7 evaluation_completion.csv exists but has no manifest_hash column."
        )
    completion_hashes = sorted(
        completion_record["manifest_hash"].dropna().astype(str).unique().tolist()
    )
    if completion_hashes != [MANIFEST_HASH]:
        raise RuntimeError(
            "Notebook-7 completion and notebook-8 compact metrics refer to different runs: "
            f"completion={completion_hashes}, notebook8={[MANIFEST_HASH]}."
        )
    NOTEBOOK7_COMPLETION_VERIFIED = True
else:
    warnings.warn(
        "Notebook-7 evaluation_completion.csv was not found. Notebook 09 will rely on "
        "the internally consistent hash stored by notebook 8.",
        RuntimeWarning,
    )

MANIFEST_MATCHES_CURRENT = MANIFEST_HASH == CURRENT_MANIFEST_HASH
if not MANIFEST_MATCHES_CURRENT:
    mismatch_message = (
        "Current validation_manifest.json differs from the frozen notebook-7/8 result run. "
        "Notebook 09 will preserve the frozen result hash and will verify scientific "
        "compatibility by reproducing the frozen LinearProxy metrics from the current "
        "reference/proxy/cohort contract files. "
        f"current={CURRENT_MANIFEST_HASH}, frozen={MANIFEST_HASH}."
    )
    if STRICT_MANIFEST_MATCH:
        raise RuntimeError(mismatch_message)
    warnings.warn(mismatch_message, RuntimeWarning)
    print("WARNING:", mismatch_message)

# Only the multigene models are consumed from notebook 8.  LinearProxy remains in
# primary_metrics solely as the frozen scientific compatibility reference below.
required_models = {"LinearProxy", "ElasticNet", "XGBoost"}
observed_models = set(primary_metrics["model"].dropna().astype(str))
missing_models = required_models - observed_models
if missing_models:
    raise RuntimeError(
        "Notebook-8 compact metrics are missing required frozen models: "
        f"{sorted(missing_models)}"
    )

multigene_metrics = primary_metrics[
    primary_metrics["model"].isin(["ElasticNet", "XGBoost"])
].copy()

print("Current manifest hash:", CURRENT_MANIFEST_HASH)
print("Frozen notebook-7/8 result hash:", MANIFEST_HASH)
print("Manifest hashes match:", MANIFEST_MATCHES_CURRENT)
print("Notebook-7 completion hash verified:", NOTEBOOK7_COMPLETION_VERIFIED)
print("External cohorts:", len(cohort_contracts))
print("CCLE proteins:", len(reference["protein_columns"]))
print(
    "Same-gene proxy eligible:",
    int(proxy_parameters["same_gene_available"].fillna(False).sum()),
)
print("Notebook-8 ENet/XGB compact rows:", f"{len(multigene_metrics):,}")


## 2. Reconstruct all three CCLE-trained proxy formulations, one cohort at a time

For a protein $p$ with cognate transcript $x_{ip}$, the three predictions are

\[
\hat y^{
m raw}_{ip}=x_{ip},
\]

\[
\hat y^{z}_{ip}=\mu^{y}_{p,
m train}+
\sigma^{y}_{p,
m train}


rac{x_{ip}-\mu^{x}_{p,
m train}}{\sigma^{x}_{p,
m train}},
\]

and

\[
\hat y^{
m lin}_{ip}=\hateta_{0p}+\hateta_{1p}x_{ip}.
\]

All means, standard deviations, slopes, and intercepts come from the **295 CCLE training samples**. External protein measurements are used only for evaluation. The calculation is cohort-by-cohort and discards arrays immediately after compact metrics are produced.

In [ ]:
protein_order = pd.Index(reference["protein_columns"]).astype(str)
rna_columns = pd.Index(reference["rna_columns"]).astype(str)
rna_median = pd.Series(np.asarray(reference["rna_median"], dtype=np.float32), index=rna_columns)
params = proxy_parameters.copy()
params["protein"] = params["protein"].astype(str)
params = params.set_index("protein").reindex(protein_order)

# Frozen training statistics in exact protein order.
eligible = params["same_gene_available"].fillna(False).to_numpy(bool)
x_mean = pd.to_numeric(params["x_mean"], errors="coerce").to_numpy(np.float32)
x_std = pd.to_numeric(params["x_std"], errors="coerce").to_numpy(np.float32)
y_mean = pd.to_numeric(params["y_mean"], errors="coerce").to_numpy(np.float32)
y_std = pd.to_numeric(params["y_std"], errors="coerce").to_numpy(np.float32)
slope = pd.to_numeric(params["slope"], errors="coerce").to_numpy(np.float32)
intercept = pd.to_numeric(params["intercept"], errors="coerce").to_numpy(np.float32)

# If a cognate transcript is unavailable in an external cohort, the primary RNA path
# uses the corresponding CCLE-training median. Reproduce that exact behavior here.
protein_ref_median = rna_median.reindex(protein_order).to_numpy(np.float32, copy=True)
protein_ref_median[~eligible] = np.nan


def same_gene_primary_matrix(rna_raw: pd.DataFrame) -> np.ndarray:
    n = len(rna_raw)
    x = np.broadcast_to(protein_ref_median, (n, len(protein_order))).copy()
    common = protein_order.intersection(rna_raw.columns.astype(str), sort=False)
    if len(common):
        protein_pos = protein_order.get_indexer(common)
        observed = rna_raw.loc[:, common].to_numpy(dtype=np.float32, copy=False)
        fill = protein_ref_median[protein_pos][None, :]
        observed = np.where(np.isfinite(observed), observed, fill)
        x[:, protein_pos] = observed
    x[:, ~eligible] = np.nan
    return x


def proxy_prediction(x: np.ndarray, proxy_type: str) -> np.ndarray:
    if proxy_type == "RawProxy":
        return x
    if proxy_type == "ZScoreProxy":
        safe = np.isfinite(x_std) & (x_std > 0) & np.isfinite(y_mean) & np.isfinite(y_std)
        out = np.broadcast_to(y_mean, x.shape).copy().astype(np.float32)
        out[:, safe] = (
            y_mean[safe][None, :]
            + ((x[:, safe] - x_mean[safe][None, :]) / x_std[safe][None, :]) * y_std[safe][None, :]
        ).astype(np.float32)
        out[:, ~eligible] = np.nan
        return out
    if proxy_type == "LinearProxy":
        out = x * slope[None, :] + intercept[None, :]
        invalid = ~eligible | ~np.isfinite(slope) | ~np.isfinite(intercept)
        out[:, invalid] = np.nan
        return out.astype(np.float32, copy=False)
    raise ValueError(proxy_type)


proxy_metric_frames = []
for cohort in expected_cohorts:
    bundle_path = cohort_contracts[cohort]
    if not bundle_path.exists():
        raise FileNotFoundError(bundle_path)
    bundle = joblib.load(bundle_path)
    rna_raw = bundle["rna_raw"]
    y = bundle["protein_raw"].reindex(columns=protein_order).to_numpy(dtype=np.float32)
    x_same = same_gene_primary_matrix(rna_raw)

    for proxy_type in PROXY_ORDER:
        pred = proxy_prediction(x_same, proxy_type)
        frame = pv.metric_frame(y, pred, protein_order, compute_spearman=False)
        frame.insert(0, "proxy_type", proxy_type)
        frame.insert(0, "platform", str(bundle["platform"]))
        frame.insert(0, "cohort", str(cohort))
        frame.insert(0, "manifest_hash", MANIFEST_HASH)
        proxy_metric_frames.append(frame[[
            "manifest_hash", "cohort", "platform", "protein", "proxy_type",
            "n", "pearson", "cosine", "mse", "rmse", "normalized_rmse"
        ]])
        if pred is not x_same:
            del pred

    del bundle, rna_raw, y, x_same
    gc.collect()
    print(f"proxy metrics: {cohort}")

proxy_cohort_metrics = pd.concat(proxy_metric_frames, ignore_index=True)
del proxy_metric_frames

# -----------------------------------------------------------------------------
# Scientific compatibility audit for a current/frozen manifest mismatch.
#
# Notebook 09 reconstructs LinearProxy from the current notebook-6 contract files.
# Notebook 8 contains the frozen notebook-7 LinearProxy metrics.  Every frozen
# LinearProxy row must therefore be reproduced numerically here.  A failure means
# the scientific contract changed and notebook 7/8 must be rerun; we never silently
# combine new proxy inputs with old ENet/XGBoost results.
# -----------------------------------------------------------------------------
COMPATIBILITY_METRICS = ["pearson", "cosine", "mse", "rmse", "normalized_rmse"]
COMPATIBILITY_RTOL = float(os.environ.get("PAPER_VALIDATION_PROXY_COMPAT_RTOL", "1e-5"))
COMPATIBILITY_ATOL = float(os.environ.get("PAPER_VALIDATION_PROXY_COMPAT_ATOL", "1e-6"))

frozen_linear = primary_metrics.loc[
    primary_metrics["model"].eq("LinearProxy"),
    ["cohort", "platform", "protein", "n", *COMPATIBILITY_METRICS],
].copy()
rebuilt_linear = proxy_cohort_metrics.loc[
    proxy_cohort_metrics["proxy_type"].eq("LinearProxy"),
    ["cohort", "platform", "protein", "n", *COMPATIBILITY_METRICS],
].copy()

if frozen_linear.empty:
    raise RuntimeError(
        "Notebook-8 compact metrics contain no frozen LinearProxy rows; "
        "cannot verify compatibility with the current contract."
    )

compatibility_keys = ["cohort", "platform", "protein"]
for label, frame in [("frozen", frozen_linear), ("rebuilt", rebuilt_linear)]:
    if frame.duplicated(compatibility_keys).any():
        examples = frame.loc[frame.duplicated(compatibility_keys, keep=False), compatibility_keys].head()
        raise RuntimeError(
            f"Duplicate {label} LinearProxy compatibility keys detected:\n{examples.to_string(index=False)}"
        )

comparison = frozen_linear.merge(
    rebuilt_linear,
    on=compatibility_keys,
    how="left",
    suffixes=("_frozen", "_rebuilt"),
    indicator=True,
    validate="one_to_one",
)

missing_rebuilt = comparison["_merge"].ne("both")
if missing_rebuilt.any():
    examples = comparison.loc[missing_rebuilt, compatibility_keys].head(10)
    raise RuntimeError(
        "The current cohort/reference contract cannot reproduce all frozen LinearProxy rows. "
        "Rerun notebooks 7 and 8 before notebook 9. Missing examples:\n"
        + examples.to_string(index=False)
    )

compatibility_rows = []

# n is a discrete metric denominator and must match exactly after numeric coercion.
n_frozen = pd.to_numeric(comparison["n_frozen"], errors="coerce").to_numpy(float)
n_rebuilt = pd.to_numeric(comparison["n_rebuilt"], errors="coerce").to_numpy(float)
n_match = np.isclose(n_frozen, n_rebuilt, rtol=0.0, atol=0.0, equal_nan=True)
compatibility_rows.append({
    "quantity": "n",
    "rows_compared": int(len(comparison)),
    "rows_matching": int(n_match.sum()),
    "max_abs_difference": float(np.nanmax(np.abs(n_frozen - n_rebuilt)))
        if np.isfinite(n_frozen - n_rebuilt).any() else 0.0,
    "rtol": 0.0,
    "atol": 0.0,
})
if not n_match.all():
    bad = comparison.loc[~n_match, compatibility_keys + ["n_frozen", "n_rebuilt"]].head(10)
    raise RuntimeError(
        "Current contract changes the LinearProxy metric denominators relative to the frozen "
        "notebook-7 run. Rerun notebooks 7 and 8. Examples:\n" + bad.to_string(index=False)
    )

for metric in COMPATIBILITY_METRICS:
    frozen_values = pd.to_numeric(comparison[f"{metric}_frozen"], errors="coerce").to_numpy(float)
    rebuilt_values = pd.to_numeric(comparison[f"{metric}_rebuilt"], errors="coerce").to_numpy(float)
    close = np.isclose(
        frozen_values,
        rebuilt_values,
        rtol=COMPATIBILITY_RTOL,
        atol=COMPATIBILITY_ATOL,
        equal_nan=True,
    )
    finite_diff = np.abs(frozen_values - rebuilt_values)
    max_diff = float(np.nanmax(finite_diff)) if np.isfinite(finite_diff).any() else 0.0
    compatibility_rows.append({
        "quantity": metric,
        "rows_compared": int(len(comparison)),
        "rows_matching": int(close.sum()),
        "max_abs_difference": max_diff,
        "rtol": COMPATIBILITY_RTOL,
        "atol": COMPATIBILITY_ATOL,
    })
    if not close.all():
        bad_cols = compatibility_keys + [f"{metric}_frozen", f"{metric}_rebuilt"]
        bad = comparison.loc[~close, bad_cols].head(10)
        raise RuntimeError(
            f"Current contract does not reproduce frozen LinearProxy {metric} metrics. "
            "This is a scientific-contract change, not a cosmetic manifest change. "
            "Rerun notebooks 7 and 8 before notebook 9. Examples:\n"
            + bad.to_string(index=False)
        )

frozen_contract_compatibility = pd.DataFrame(compatibility_rows)
frozen_contract_compatibility.insert(0, "manifest_hash", MANIFEST_HASH)
frozen_contract_compatibility.insert(1, "current_manifest_hash", CURRENT_MANIFEST_HASH)
frozen_contract_compatibility["compatible"] = (
    frozen_contract_compatibility["rows_compared"]
    == frozen_contract_compatibility["rows_matching"]
)
CURRENT_CONTRACT_REPRODUCES_FROZEN_LINEAR_PROXY = bool(
    frozen_contract_compatibility["compatible"].all()
)

compatibility_path = pv.write_table(
    frozen_contract_compatibility,
    LAYOUT["procedural"] / "current_contract_vs_frozen_linear_proxy_compatibility",
    index=False,
)

print(
    "PASS — current contract reproduces the frozen notebook-7 LinearProxy metrics:",
    CURRENT_CONTRACT_REPRODUCES_FROZEN_LINEAR_PROXY,
)
display(frozen_contract_compatibility)

proxy_cohort_metrics_path = pv.write_table(
    proxy_cohort_metrics,
    LAYOUT["procedural"] / "best_proxy_cohort_metrics",
    index=False,
)
print("Proxy cohort-metric rows:", f"{len(proxy_cohort_metrics):,}")
display(proxy_cohort_metrics.head())


## 3. Aggregate cohort metrics within each platform

To preserve the three external platform groups without pooling all samples into one matrix:

- **Pearson** correlations are combined with a Fisher $z$ transform weighted by $n-3$;
- **cosine similarity** and **MSE** are aggregated across cohorts with paired-sample-count weights.

This produces one compact value for each platform × protein × proxy/model. The same platform-level values are used for best-proxy selection, the Pearson/cosine cascades, and the MSE quantile summaries.

In [ ]:
def aggregate_metric_by_platform(frame: pd.DataFrame, metric: str, identity_col: str) -> pd.DataFrame:
    """One value per platform x protein x model/proxy identity."""
    work = frame[["platform", "protein", identity_col, "cohort", "n", metric]].copy()
    work[metric] = pd.to_numeric(work[metric], errors="coerce")
    work["n"] = pd.to_numeric(work["n"], errors="coerce")
    rows = []
    for key, g in work.groupby(["platform", "protein", identity_col], observed=True, sort=False):
        platform, protein, identity = key
        vals = g[metric].to_numpy(float)
        ns = g["n"].to_numpy(float)
        if metric == "pearson":
            valid = np.isfinite(vals) & np.isfinite(ns) & (ns >= 4)
            if valid.any():
                z = np.arctanh(np.clip(vals[valid], -0.999999, 0.999999))
                w = np.maximum(ns[valid] - 3.0, 1.0)
                value = float(np.tanh(np.sum(w * z) / np.sum(w)))
                total_weight = float(np.sum(w))
            else:
                value = np.nan; total_weight = 0.0
        else:
            valid = np.isfinite(vals) & np.isfinite(ns) & (ns > 0)
            if valid.any():
                w = ns[valid]
                value = float(np.sum(w * vals[valid]) / np.sum(w))
                total_weight = float(np.sum(w))
            else:
                value = np.nan; total_weight = 0.0
        rows.append({
            "platform": str(platform), "protein": str(protein), identity_col: str(identity),
            "metric": metric, "value": value,
            "cohorts_contributing": int(g.loc[valid, "cohort"].nunique()) if len(g) else 0,
            "total_weight": total_weight,
        })
    return pd.DataFrame(rows)

platform_metric_frames = []
for metric in ALL_METRICS:
    # Three proxy variants.
    p = aggregate_metric_by_platform(proxy_cohort_metrics, metric, "proxy_type")
    p = p.rename(columns={"proxy_type": "identity"})
    p["family"] = "Proxy"
    platform_metric_frames.append(p)

    # Frozen multigene models from notebook 8.
    m = aggregate_metric_by_platform(multigene_metrics, metric, "model")
    m = m.rename(columns={"model": "identity"})
    m["family"] = "Model"
    platform_metric_frames.append(m)

platform_metric_values = pd.concat(platform_metric_frames, ignore_index=True)
platform_metric_values["manifest_hash"] = MANIFEST_HASH
platform_metric_values_path = pv.write_table(
    platform_metric_values,
    LAYOUT["procedural"] / "platform_metric_values_for_best_proxy_cascade",
    index=False,
)

display(
    platform_metric_values.groupby(["metric", "platform", "family", "identity"], observed=True)
    .agg(proteins=("protein", "nunique"), median_value=("value", "median"))
    .reset_index()
)

## 4. Select the best proxy once; threshold Pearson/cosine only

For Pearson and cosine similarity, the best proxy is the one with the largest platform-level metric; for MSE, it is the one with the smallest MSE. The comparison universe is restricted to proteins with finite values for all three proxy formulations and both multigene models.

Only Pearson and cosine enter a hard-threshold cascade. **MSE does not.** MSE is summarized later by quantiles on the same matched protein universe.

In [ ]:
def select_best_proxy(platform_values: pd.DataFrame, metric: str, direction: str) -> pd.DataFrame:
    sub = platform_values[
        (platform_values["metric"].eq(metric))
        & (platform_values["family"].eq("Proxy"))
        & (platform_values["identity"].isin(PROXY_ORDER))
    ].copy()
    pivot = sub.pivot_table(index=["platform", "protein"], columns="identity", values="value", aggfunc="first")
    for proxy in PROXY_ORDER:
        if proxy not in pivot:
            pivot[proxy] = np.nan
    # Exact comparable proxy universe: all three variants must have finite values.
    pivot = pivot.dropna(subset=PROXY_ORDER).copy()
    if direction == "higher":
        winner = pivot[PROXY_ORDER].idxmax(axis=1)
        best = pivot[PROXY_ORDER].max(axis=1)
    else:
        winner = pivot[PROXY_ORDER].idxmin(axis=1)
        best = pivot[PROXY_ORDER].min(axis=1)
    out = pivot.reset_index()[["platform", "protein"]]
    out["metric"] = metric
    out["best_proxy_type"] = winner.to_numpy()
    out["best_proxy_value"] = best.to_numpy(float)
    return out


def matched_metric_values(platform_values: pd.DataFrame, metric: str) -> pd.DataFrame:
    """Best proxy + ENet + XGB on one exact matched protein universe."""
    best_proxy = select_best_proxy(platform_values, metric, METRIC_DIRECTION[metric])
    model_sub = platform_values[
        (platform_values["metric"].eq(metric))
        & (platform_values["family"].eq("Model"))
        & (platform_values["identity"].isin(["ElasticNet", "XGBoost"]))
    ].copy()
    model_pivot = model_sub.pivot_table(
        index=["platform", "protein"], columns="identity", values="value", aggfunc="first"
    ).reset_index()
    for model in ["ElasticNet", "XGBoost"]:
        if model not in model_pivot:
            model_pivot[model] = np.nan

    values = best_proxy.merge(model_pivot, on=["platform", "protein"], how="inner", validate="one_to_one")
    values = values.dropna(subset=["best_proxy_value", "ElasticNet", "XGBoost"]).copy()
    values = values[values["platform"].isin(PLATFORM_ORDER)].copy()
    return values


def build_higher_is_better_cascade(values: pd.DataFrame, metric: str, thresholds) -> tuple[pd.DataFrame, pd.DataFrame]:
    assignment_frames = []
    summary_rows = []
    for threshold in thresholds:
        tmp = values.copy()
        proxy_ok = tmp["best_proxy_value"] >= threshold
        enet_ok = (~proxy_ok) & (tmp["ElasticNet"] >= threshold)
        xgb_ok = (~proxy_ok) & (~enet_ok) & (tmp["XGBoost"] >= threshold)

        tmp["threshold"] = float(threshold)
        tmp["stage"] = np.select(
            [proxy_ok, enet_ok, xgb_ok],
            ["Best proxy", "ElasticNet", "XGBoost"],
            default="Unresolved",
        )
        tmp["metric"] = metric
        assignment_frames.append(tmp)

        for platform in PLATFORM_ORDER:
            g = tmp[tmp["platform"].eq(platform)]
            denominator = int(len(g))
            for stage in STAGE_ORDER:
                count = int(g["stage"].eq(stage).sum())
                summary_rows.append({
                    "metric": metric,
                    "platform": platform,
                    "threshold": float(threshold),
                    "stage": stage,
                    "proteins": count,
                    "denominator": denominator,
                    "percent": 100.0 * count / denominator if denominator else np.nan,
                })

    return pd.concat(assignment_frames, ignore_index=True), pd.DataFrame(summary_rows)


# One matched table per metric. MSE is retained here for best-proxy selection and quantiles,
# but is deliberately excluded from the hard-threshold cascade.
matched_frames = [matched_metric_values(platform_metric_values, metric) for metric in ALL_METRICS]
best_proxy_selection = pd.concat(matched_frames, ignore_index=True)
best_proxy_selection["manifest_hash"] = MANIFEST_HASH

assignment_frames = []
summary_frames = []
for metric, spec in CASCADE_SPECS.items():
    values = best_proxy_selection[best_proxy_selection["metric"].eq(metric)].drop(columns="manifest_hash")
    assignments, summary = build_higher_is_better_cascade(values, metric, spec["thresholds"])
    assignment_frames.append(assignments)
    summary_frames.append(summary)

cascade_assignments = pd.concat(assignment_frames, ignore_index=True)
cascade_summary = pd.concat(summary_frames, ignore_index=True)
for frame in (cascade_assignments, cascade_summary):
    frame["manifest_hash"] = MANIFEST_HASH

# Threshold-free MSE quantiles on the exact matched best-proxy/ENet/XGB universe.
mse_values = best_proxy_selection[best_proxy_selection["metric"].eq("mse")].copy()
mse_long = pd.concat([
    mse_values[["platform", "protein", "best_proxy_type", "best_proxy_value"]]
        .rename(columns={"best_proxy_value": "mse"}).assign(model="Best proxy"),
    mse_values[["platform", "protein", "best_proxy_type", "ElasticNet"]]
        .rename(columns={"ElasticNet": "mse"}).assign(model="ElasticNet"),
    mse_values[["platform", "protein", "best_proxy_type", "XGBoost"]]
        .rename(columns={"XGBoost": "mse"}).assign(model="XGBoost"),
], ignore_index=True)

mse_quantile_rows = []
for (platform, model), g in mse_long.groupby(["platform", "model"], observed=True, sort=False):
    vals = pd.to_numeric(g["mse"], errors="coerce").to_numpy(float)
    vals = vals[np.isfinite(vals)]
    if not len(vals):
        continue
    row = {
        "platform": str(platform),
        "model": str(model),
        "proteins": int(len(vals)),
    }
    for q in MSE_QUANTILES:
        row[f"q{int(round(100*q)):02d}"] = float(np.quantile(vals, q))
    mse_quantile_rows.append(row)

mse_quantile_summary = pd.DataFrame(mse_quantile_rows)
mse_quantile_summary["manifest_hash"] = MANIFEST_HASH

best_proxy_selection_path = pv.write_table(
    best_proxy_selection, LAYOUT["procedural"] / "best_proxy_selection_by_metric_platform", index=False
)
cascade_assignments_path = pv.write_table(
    cascade_assignments, LAYOUT["procedural"] / "best_proxy_cascade_assignments_pearson_cosine", index=False
)
cascade_summary_path = pv.write_table(
    cascade_summary, LAYOUT["procedural"] / "best_proxy_cascade_summary_pearson_cosine", index=False
)
mse_quantile_summary_path = pv.write_table(
    mse_quantile_summary, LAYOUT["procedural"] / "best_proxy_mse_quantiles", index=False
)

print("Pearson/cosine cascade counts")
display(
    cascade_summary.pivot_table(
        index=["metric", "platform", "threshold"], columns="stage", values="proteins",
        fill_value=0, aggfunc="sum"
    ).reindex(columns=STAGE_ORDER).reset_index()
)
print("MSE quantiles — no hard thresholds")
display(mse_quantile_summary.sort_values(["platform", "model"]))

## 5. Which of the three proxy formulations wins?

This companion summary reports the proxy formulation selected before the procedural analysis. For Pearson/cosine the maximum metric wins; for MSE the minimum MSE wins. The selection is descriptive/oracle-style because it uses the external metric.

In [ ]:
best_proxy_composition = (
    best_proxy_selection.groupby(["metric", "platform", "best_proxy_type"], observed=True)
    .size().rename("proteins").reset_index()
)
best_proxy_composition["denominator"] = best_proxy_composition.groupby(
    ["metric", "platform"], observed=True
)["proteins"].transform("sum")
best_proxy_composition["percent"] = 100.0 * best_proxy_composition["proteins"] / best_proxy_composition["denominator"]
best_proxy_composition["manifest_hash"] = MANIFEST_HASH
best_proxy_composition_path = pv.write_table(
    best_proxy_composition, LAYOUT["procedural"] / "best_proxy_composition", index=False
)
display(best_proxy_composition.sort_values(["metric", "platform", "proteins"], ascending=[True, True, False]))

## 6. Paper-ready figures — PDF only and enlarged typography

All figures are exported directly as vector PDFs into `figures_pdf`. Typography is doubled relative to the previous plotting configuration, and the figure geometry is enlarged at the same time so labels and legends remain readable without clipping.

For Pearson, the three platform cascades retain the established filenames `cascade_tmt_v1.pdf`, `cascade_itraq_v1.pdf`, and `cascade_labelfree_v1.pdf`. A fourth figure, `cascade_platform_pcc_v1.pdf`, displays TMT, iTRAQ, and Label-free together with one shared legend. No PNG intermediary is created or required.

Pearson and cosine retain the cascade **Best same-gene proxy → ElasticNet → XGBoost → Unresolved**. MSE remains a quantile analysis only; there is no hard-threshold MSE cascade.

In [ ]:
# -----------------------------------------------------------------------------
# Paper-ready figures: PDF only, doubled typography, and a combined platform view
# -----------------------------------------------------------------------------
FIG_DPI = int(os.environ.get("PAPER_VALIDATION_FIG_DPI", "300"))
FIG_META = {"Software": pv.PIPELINE_VERSION, "ManifestHash": MANIFEST_HASH}

import math
from matplotlib import colors as mcolors
from matplotlib.ticker import MaxNLocator

# Every figure from this notebook is written here as PDF only.
FIGURES_PDF = FIG_DIR
FIGURES_PDF.mkdir(parents=True, exist_ok=True)

# Keep text editable in vector-PDF workflows.
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 23,
    "axes.labelsize": 20,
    "xtick.labelsize": 19,
    "ytick.labelsize": 19,
    "legend.fontsize": 18,
    "figure.titlesize": 25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

STAGE_COLORS = {
    "Best proxy": "#4C78A8",
    "ElasticNet": "#54A24B",
    "XGBoost": "#F58518",
    "Unresolved": "#E45756",
}
MODEL_COLORS = {
    "Best proxy": "#4C78A8",
    "ElasticNet": "#54A24B",
    "XGBoost": "#F58518",
}
PROXY_COLORS = {
    "RawProxy": "#4C78A8",
    "ZScoreProxy": "#B279A2",
    "LinearProxy": "#54A24B",
}
PROXY_LABELS = {
    "RawProxy": "Raw RNA proxy",
    "ZScoreProxy": "Train-z proxy",
    "LinearProxy": "Train-linear proxy",
}
CASCADE_LEGEND_LABELS = {
    "Best proxy": "Best same-gene proxy — reaches the threshold first",
    "ElasticNet": "ElasticNet — among proteins remaining after the proxy",
    "XGBoost": "XGBoost — among proteins remaining after proxy + ElasticNet",
    "Unresolved": "Unresolved — no tested model reaches the threshold",
}

# Single source of truth for the final provenance registry: PDF paths only.
figure_paths = []


def _save_pdf(fig, file_stem, title):
    """Save one vector PDF to figures_pdf and register it."""
    path = FIGURES_PDF / f"{file_stem}.pdf"
    fig.savefig(
        path,
        format="pdf",
        bbox_inches="tight",
        pad_inches=0.30,
        metadata={"Title": title, "Subject": MANIFEST_HASH},
    )
    figure_paths.append(path)
    return path


def _platform_display_name(platform):
    return "Label-free" if str(platform) == "LabelFree" else str(platform)


def _platform_file_name(platform):
    return str(platform).lower().replace(" ", "_").replace("-", "_")


def _contrasting_text_color(color):
    red, green, blue = mcolors.to_rgb(color)
    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    return "#111111" if luminance > 0.68 else "white"


def _cascade_summary_for_platform(metric, platform, thresholds):
    thresholds = [float(x) for x in thresholds]
    sub = cascade_summary[
        cascade_summary["metric"].eq(metric)
        & cascade_summary["platform"].eq(platform)
    ].copy()
    sub["threshold"] = pd.to_numeric(sub["threshold"], errors="coerce")
    sub["proteins"] = pd.to_numeric(sub["proteins"], errors="coerce").fillna(0)
    pivot = (
        sub.pivot_table(
            index="threshold",
            columns="stage",
            values="proteins",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(index=thresholds, columns=STAGE_ORDER, fill_value=0)
        .astype(float)
    )
    if pivot.empty or float(pivot.to_numpy().sum()) <= 0:
        raise RuntimeError(f"No cascade counts for metric={metric}, platform={platform}.")
    return pivot


def _draw_stacked_cascade_axis(ax, metric, platform, thresholds, row_label_func, *, show_xlabel=True):
    """Draw one platform cascade without creating a legend (shared legends are added by caller)."""
    pivot = _cascade_summary_for_platform(metric, platform, thresholds)
    thresholds = list(pivot.index)
    n_rows = len(thresholds)
    y_positions = np.arange(n_rows, dtype=float) * 1.75
    left = np.zeros(n_rows, dtype=float)
    totals = pivot.sum(axis=1).to_numpy(dtype=float)
    max_total = max(float(np.nanmax(totals)) if len(totals) else 1.0, 1.0)
    bar_height = 0.96

    # Stagger tiny-segment labels vertically so they remain readable at doubled font size.
    tiny_offsets = (-0.72, 0.72, -1.02, 1.02)

    for stage_index, stage in enumerate(STAGE_ORDER):
        values = pivot[stage].to_numpy(dtype=float)
        color = STAGE_COLORS[stage]
        bars = ax.barh(
            y_positions,
            values,
            left=left,
            height=bar_height,
            color=color,
            edgecolor="white",
            linewidth=1.8,
            zorder=3,
        )

        for row_index, (bar, value, total) in enumerate(zip(bars, values, totals)):
            if value <= 0 or total <= 0:
                continue
            percent = 100.0 * value / total
            centre = left[row_index] + 0.5 * value
            if percent >= 7.0:
                ax.text(
                    centre,
                    y_positions[row_index],
                    f"{int(round(value)):,}\n{percent:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=20,
                    fontweight="bold",
                    color=_contrasting_text_color(color),
                    linespacing=1.05,
                    zorder=5,
                )
            else:
                offset = tiny_offsets[stage_index % len(tiny_offsets)]
                ax.annotate(
                    f"{int(round(value)):,}",
                    xy=(centre, y_positions[row_index]),
                    xytext=(centre, y_positions[row_index] + offset),
                    ha="center",
                    va="center",
                    fontsize=16.5,
                    fontweight="bold",
                    color="#2F2F2F",
                    arrowprops={
                        "arrowstyle": "-",
                        "color": color,
                        "linewidth": 1.6,
                        "shrinkA": 2,
                        "shrinkB": 2,
                    },
                    clip_on=False,
                    zorder=6,
                )
        left += values

    ax.set_yticks(y_positions)
    ax.set_yticklabels([row_label_func(t) for t in thresholds], fontsize=20)
    ax.invert_yaxis()
    if show_xlabel:
        ax.set_xlabel("Number of proteins", fontsize=21, labelpad=12)
    ax.set_xlim(0, max_total)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
    ax.tick_params(axis="x", labelsize=18, pad=7)
    ax.tick_params(axis="y", length=0, pad=12)
    ax.grid(axis="x", alpha=0.20, linewidth=1.0, zorder=0)
    ax.grid(axis="y", visible=False)
    ax.spines["left"].set_visible(False)
    ax.set_title(_platform_display_name(platform), loc="left", fontsize=24, fontweight="bold", pad=14)
    return pivot


def _cascade_handles():
    return [
        Patch(facecolor=STAGE_COLORS[stage], edgecolor="white", label=CASCADE_LEGEND_LABELS[stage])
        for stage in STAGE_ORDER
    ]


def plot_individual_stacked_cascade(metric, platform, thresholds, row_label_func, file_stem):
    """Large standalone cascade with the same paper-ready styling."""
    fig, ax = plt.subplots(figsize=(22, 9.5))
    _draw_stacked_cascade_axis(ax, metric, platform, thresholds, row_label_func)
    fig.legend(
        handles=_cascade_handles(),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=2,
        frameon=False,
        fontsize=17,
        handlelength=1.7,
        columnspacing=2.2,
        labelspacing=0.9,
    )
    fig.subplots_adjust(left=0.17, right=0.985, top=0.89, bottom=0.25)
    path = _save_pdf(
        fig,
        file_stem,
        f"{metric.title()} cascade — {_platform_display_name(platform)}",
    )
    plt.show()
    plt.close(fig)
    return path


def plot_combined_pearson_platforms():
    """TMT + iTRAQ + Label-free Pearson cascades in one figure with one shared legend."""
    fig, axes = plt.subplots(3, 1, figsize=(23, 23), constrained_layout=False)
    for idx, (ax, platform) in enumerate(zip(axes, PLATFORM_ORDER)):
        _draw_stacked_cascade_axis(
            ax,
            "pearson",
            platform,
            PEARSON_THRESHOLDS,
            lambda t: f"Pearson >= {t:.2f}",
            show_xlabel=(idx == len(PLATFORM_ORDER) - 1),
        )
    fig.legend(
        handles=_cascade_handles(),
        loc="lower center",
        bbox_to_anchor=(0.5, 0.012),
        ncol=2,
        frameon=False,
        fontsize=18,
        handlelength=1.8,
        columnspacing=2.6,
        labelspacing=1.0,
    )
    fig.subplots_adjust(left=0.17, right=0.985, top=0.975, bottom=0.15, hspace=0.40)
    path = _save_pdf(fig, "cascade_platform_pcc_v1", "Pearson cascade — TMT, iTRAQ and Label-free")
    plt.show()
    plt.close(fig)
    return path


# -----------------------------------------------------------------------------
# Pearson: keep the three established LaTeX filenames, plus a combined view.
# -----------------------------------------------------------------------------
pearson_file_stems = {
    "TMT": "cascade_tmt_v1",
    "iTRAQ": "cascade_itraq_v1",
    "LabelFree": "cascade_labelfree_v1",
}
stacked_cascade_paths = {}
for platform in PLATFORM_ORDER:
    stacked_cascade_paths[("pearson", platform)] = plot_individual_stacked_cascade(
        "pearson",
        platform,
        PEARSON_THRESHOLDS,
        lambda t: f"Pearson >= {t:.2f}",
        pearson_file_stems[platform],
    )

combined_pcc_pdf = plot_combined_pearson_platforms()


# -----------------------------------------------------------------------------
# Cosine cascades: PDF only, same enlarged formatting.
# -----------------------------------------------------------------------------
for platform in PLATFORM_ORDER:
    platform_file = _platform_file_name(platform)
    stacked_cascade_paths[("cosine", platform)] = plot_individual_stacked_cascade(
        "cosine",
        platform,
        COSINE_THRESHOLDS,
        lambda t: f"Cosine >= {t:.2f}",
        f"figure_external_cosine_{platform_file}_cascade_notebook4_stacked",
    )


# -----------------------------------------------------------------------------
# Alternative flow views: PDF only; fonts and geometry enlarged together.
# -----------------------------------------------------------------------------
def _format_flow_percent(value):
    value = float(value)
    if 0 < abs(value) < 0.1:
        return f"{value:.2f}%"
    return f"{value:.1f}%"


def _flow_box(ax, x, y, width, height, text, *, facecolor, edgecolor="white", fontsize=27, textcolor=None):
    from matplotlib.patches import FancyBboxPatch
    patch = FancyBboxPatch(
        (x - width / 2.0, y - height / 2.0),
        width,
        height,
        boxstyle="round,pad=0.014,rounding_size=0.016",
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=2.2,
        zorder=3,
    )
    ax.add_patch(patch)
    ax.text(
        x, y, text,
        ha="center", va="center",
        fontsize=fontsize, fontweight="bold",
        color=textcolor or _contrasting_text_color(facecolor),
        zorder=4, linespacing=1.10,
    )
    return patch


def _flow_arrow(ax, x1, y1, x2, y2, *, label=None, label_y_offset=0.27):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops={
            "arrowstyle": "-|>", "linewidth": 2.5, "color": "#666666",
            "shrinkA": 4, "shrinkB": 4, "mutation_scale": 22,
        },
        zorder=2,
    )
    if label:
        ax.text(
            (x1 + x2) / 2.0,
            (y1 + y2) / 2.0 + label_y_offset,
            label,
            ha="center", va="bottom",
            fontsize=21, fontweight="medium", color="#555555",
            zorder=5,
        )


def plot_flow_cascade(metric, platform, thresholds, row_label_func, file_stem):
    pivot = _cascade_summary_for_platform(metric, platform, thresholds)
    thresholds = list(pivot.index)
    n_rows = len(thresholds)
    fig, axes = plt.subplots(n_rows, 1, figsize=(28, max(13, 4.2 * n_rows)), squeeze=False)
    axes = axes[:, 0]
    positions = np.linspace(0.25, 0.94, 5)
    box_w = 0.14
    box_h = 0.40
    headers = ["All proteins", "Best same-gene\nproxy", "ElasticNet", "XGBoost", "Unresolved"]

    for row_i, (ax, threshold) in enumerate(zip(axes, thresholds)):
        row = pivot.loc[threshold]
        total = max(float(row.sum()), 1.0)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")
        ax.text(
            0.005, 0.50, row_label_func(threshold),
            ha="left", va="center", fontsize=27, fontweight="bold",
            color="#202020", clip_on=False,
        )
        if row_i == 0:
            for x, header in zip(positions, headers):
                ax.text(x, 1.12, header, ha="center", va="bottom", fontsize=24,
                        fontweight="bold", color="#252525", linespacing=1.0, clip_on=False)

        _flow_box(ax, positions[0], 0.50, box_w, box_h,
                  f"{int(round(total)):,}\nproteins", facecolor="#F1F1F1",
                  edgecolor="#A8A8A8", textcolor="#2A2A2A", fontsize=25)
        remaining = total
        previous_x = positions[0]
        for idx, stage in enumerate(["Best proxy", "ElasticNet", "XGBoost"], start=1):
            value = float(row.get(stage, 0.0))
            input_n = max(remaining, 0.0)
            pct_input = 100.0 * value / input_n if input_n > 0 else 0.0
            stage_x = positions[idx]
            _flow_arrow(ax, previous_x + box_w/2, 0.50, stage_x - box_w/2, 0.50)
            _flow_box(ax, stage_x, 0.50, box_w, box_h,
                      f"{int(round(value)):,}\n{_format_flow_percent(pct_input)} of input",
                      facecolor=STAGE_COLORS[stage], fontsize=24)
            remaining = max(0.0, remaining - value)
            next_x = positions[idx + 1]
            _flow_arrow(ax, stage_x + box_w/2, 0.50, next_x - box_w/2, 0.50,
                        label=f"{int(round(remaining)):,} remaining")
            previous_x = stage_x

        unresolved_n = float(row.get("Unresolved", remaining))
        _flow_box(ax, positions[-1], 0.50, box_w, box_h,
                  f"{int(round(unresolved_n)):,}\nunresolved",
                  facecolor=STAGE_COLORS["Unresolved"], fontsize=24)

    fig.suptitle(_platform_display_name(platform), x=0.055, ha="left", fontsize=27, fontweight="bold", y=0.985)
    fig.subplots_adjust(left=0.055, right=0.99, top=0.90, bottom=0.055, hspace=0.50)
    path = _save_pdf(fig, file_stem, f"{metric.title()} cascade flow — {_platform_display_name(platform)}")
    plt.show()
    plt.close(fig)
    return path


flow_cascade_paths = {}
for metric, thresholds, row_label_func in [
    ("pearson", PEARSON_THRESHOLDS, lambda t: f"Pearson >= {t:.2f}"),
    ("cosine", COSINE_THRESHOLDS, lambda t: f"Cosine >= {t:.2f}"),
]:
    for platform in PLATFORM_ORDER:
        platform_file = _platform_file_name(platform)
        flow_cascade_paths[(metric, platform)] = plot_flow_cascade(
            metric,
            platform,
            thresholds,
            row_label_func,
            f"figure_external_{metric}_{platform_file}_cascade_notebook4_flow",
        )


# -----------------------------------------------------------------------------
# MSE quantiles: PDF only, larger drawing and doubled text.
# -----------------------------------------------------------------------------
qcols = {int(round(100*q)): f"q{int(round(100*q)):02d}" for q in MSE_QUANTILES}
required_q = {10, 25, 50, 75, 90}
if not required_q.issubset(qcols):
    raise RuntimeError("The paper-ready MSE plot requires q10, q25, q50, q75, and q90.")

fig, axes = plt.subplots(1, 3, figsize=(28, 9.5), constrained_layout=False)
model_display_order = ["Best proxy", "ElasticNet", "XGBoost"]
for ax, platform in zip(axes, PLATFORM_ORDER):
    sub = mse_quantile_summary[mse_quantile_summary["platform"].eq(platform)].set_index("model")
    y = np.arange(len(model_display_order))
    for yi, model in enumerate(model_display_order):
        if model not in sub.index:
            continue
        row = sub.loc[model]
        q10 = float(row[qcols[10]]); q25 = float(row[qcols[25]])
        q50 = float(row[qcols[50]]); q75 = float(row[qcols[75]]); q90 = float(row[qcols[90]])
        color = MODEL_COLORS[model]
        ax.hlines(yi, q10, q90, color=color, linewidth=4.0, alpha=0.65)
        ax.hlines(yi, q25, q75, color=color, linewidth=15.0, alpha=0.95)
        ax.scatter([q50], [yi], s=190, color="white", edgecolor=color, linewidth=3.2, zorder=3)
    ax.set_yticks(y, model_display_order)
    ax.invert_yaxis()
    ax.set_xlabel("MSE (lower is better)", labelpad=12)
    ax.set_title(_platform_display_name(platform), fontweight="bold", pad=14)
    ax.grid(axis="x", alpha=0.20, linewidth=1.0)
    ax.set_xlim(left=0)
    ax.tick_params(axis="both", pad=8)

quantile_legend = [
    Line2D([0], [0], color="#555555", lw=4, label="10th–90th percentile"),
    Line2D([0], [0], color="#555555", lw=15, label="25th–75th percentile"),
    Line2D([0], [0], marker="o", markersize=12, markerfacecolor="white", markeredgecolor="#555555",
           markeredgewidth=2.4, lw=0, label="Median"),
]
fig.legend(handles=quantile_legend, loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=3, frameon=False)
fig.suptitle("External MSE distribution by model — quantiles only, no hard thresholds",
             fontsize=25, fontweight="bold", y=0.985)
fig.subplots_adjust(bottom=0.20, top=0.83, left=0.07, right=0.985, wspace=0.36)
mse_pdf = _save_pdf(fig, "figure_external_mse_quantiles", "External MSE quantiles")
plt.show()
plt.close(fig)


# -----------------------------------------------------------------------------
# Best-proxy identity composition: PDF only, larger drawing and doubled text.
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(28, 9.5), constrained_layout=False, sharex=True)
for ax, metric in zip(axes, ["pearson", "cosine", "mse"]):
    sub = best_proxy_composition[best_proxy_composition["metric"].eq(metric)]
    y = np.arange(len(PLATFORM_ORDER))
    left = np.zeros(len(PLATFORM_ORDER), dtype=float)
    denominators = (
        sub.groupby("platform", observed=True)["denominator"].max()
        .reindex(PLATFORM_ORDER).fillna(0).to_numpy(float)
    )
    for proxy in PROXY_ORDER:
        counts = (
            sub[sub["best_proxy_type"].eq(proxy)].set_index("platform")
            .reindex(PLATFORM_ORDER)["proteins"].fillna(0).to_numpy(float)
        )
        pcts = np.divide(100*counts, denominators, out=np.zeros_like(counts), where=denominators>0)
        bars = ax.barh(y, pcts, left=left, height=0.68, color=PROXY_COLORS[proxy],
                       edgecolor="white", linewidth=1.4)
        for bar, count, pct in zip(bars, counts, pcts):
            if pct >= 11 and count > 0:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+bar.get_height()/2,
                        f"{int(count)}\n{pct:.0f}%", ha="center", va="center",
                        fontsize=16, color="white", fontweight="bold")
        left += pcts
    ax.set_yticks(y, [_platform_display_name(p) for p in PLATFORM_ORDER])
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Proteins (%)", labelpad=12)
    ax.set_title({"pearson":"Pearson", "cosine":"Cosine similarity", "mse":"MSE"}[metric],
                 fontweight="bold", pad=14)
    ax.grid(axis="x", alpha=0.20, linewidth=1.0)
    ax.tick_params(axis="both", pad=8)
proxy_legend = [Patch(facecolor=PROXY_COLORS[p], edgecolor="none", label=PROXY_LABELS[p]) for p in PROXY_ORDER]
fig.legend(handles=proxy_legend, loc="lower center", bbox_to_anchor=(0.5, 0.02), ncol=3, frameon=False)
fig.suptitle("Best same-gene proxy formulation selected before model comparison",
             fontsize=25, fontweight="bold", y=0.985)
fig.subplots_adjust(bottom=0.20, top=0.83, left=0.08, right=0.985, wspace=0.32)
proxy_pdf = _save_pdf(fig, "figure_external_best_proxy_identity_notebook4_style", "Best external proxy formulation")
plt.show()
plt.close(fig)

print(f"Saved {len(figure_paths)} PDF figures to: {FIGURES_PDF}")


In [ ]:
# -----------------------------------------------------------------------------
# Combined V2 Pearson cascade flow view: TMT + iTRAQ + Label-free, one legend
# -----------------------------------------------------------------------------
# This is generated directly from cascade_summary; it does not read or depend on
# any PNG/PDF produced elsewhere in the notebook.


def _draw_platform_flow_row(
    ax,
    *,
    row,
    threshold,
    row_label_func,
    show_headers=False,
    platform_title=None,
):
    positions = np.linspace(0.25, 0.94, 5)
    box_w = 0.135
    box_h = 0.42
    headers = ["All proteins", "Best same-gene\nproxy", "ElasticNet", "XGBoost", "Unresolved"]

    total = max(float(row.sum()), 1.0)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    if platform_title:
        ax.text(
            0.0, 1.42, platform_title,
            transform=ax.transAxes,
            ha="left", va="bottom",
            fontsize=29, fontweight="bold",
            color="#202020", clip_on=False,
        )

    ax.text(
        0.005, 0.50, row_label_func(threshold),
        ha="left", va="center",
        fontsize=22, fontweight="bold",
        color="#202020", clip_on=False,
    )

    if show_headers:
        for x, header in zip(positions, headers):
            ax.text(
                x, 1.10, header,
                ha="center", va="bottom",
                fontsize=21, fontweight="bold",
                color="#252525", linespacing=1.0,
                clip_on=False,
            )

    _flow_box(
        ax, positions[0], 0.50, box_w, box_h,
        f"{int(round(total)):,}\nproteins",
        facecolor="#F1F1F1", edgecolor="#A8A8A8",
        textcolor="#2A2A2A", fontsize=20,
    )

    remaining = total
    previous_x = positions[0]
    for idx, stage in enumerate(["Best proxy", "ElasticNet", "XGBoost"], start=1):
        value = float(row.get(stage, 0.0))
        input_n = max(remaining, 0.0)
        pct_input = 100.0 * value / input_n if input_n > 0 else 0.0
        stage_x = positions[idx]

        _flow_arrow(ax, previous_x + box_w/2, 0.50, stage_x - box_w/2, 0.50)
        _flow_box(
            ax, stage_x, 0.50, box_w, box_h,
            f"{int(round(value)):,}\n{_format_flow_percent(pct_input)} of input",
            facecolor=STAGE_COLORS[stage], fontsize=19,
        )

        remaining = max(0.0, remaining - value)
        next_x = positions[idx + 1]
        _flow_arrow(
            ax, stage_x + box_w/2, 0.50, next_x - box_w/2, 0.50,
            label=f"{int(round(remaining)):,} remaining",
        )
        previous_x = stage_x

    unresolved_n = float(row.get("Unresolved", remaining))
    _flow_box(
        ax, positions[-1], 0.50, box_w, box_h,
        f"{int(round(unresolved_n)):,}\nunresolved",
        facecolor=STAGE_COLORS["Unresolved"], fontsize=19,
    )


_v2_platforms = ["TMT", "iTRAQ", "LabelFree"]
_v2_thresholds = [0.30, 0.50, 0.70]
fig_platform_v2, axes_platform_v2 = plt.subplots(
    len(_v2_platforms) * len(_v2_thresholds), 1,
    figsize=(30, 34),
    squeeze=False,
)
axes_platform_v2 = axes_platform_v2[:, 0]

axis_i = 0
for platform in _v2_platforms:
    pivot = _cascade_summary_for_platform("pearson", platform, _v2_thresholds)
    for row_i, threshold in enumerate(pivot.index):
        _draw_platform_flow_row(
            axes_platform_v2[axis_i],
            row=pivot.loc[threshold],
            threshold=threshold,
            row_label_func=lambda t: f"Pearson >= {t:.2f}",
            show_headers=(row_i == 0),
            platform_title=(_platform_display_name(platform) if row_i == 0 else None),
        )
        axis_i += 1

platform_v2_handles = [
    Patch(facecolor=STAGE_COLORS["Best proxy"], edgecolor="white", label="Best same-gene proxy"),
    Patch(facecolor=STAGE_COLORS["ElasticNet"], edgecolor="white", label="ElasticNet"),
    Patch(facecolor=STAGE_COLORS["XGBoost"], edgecolor="white", label="XGBoost"),
    Patch(facecolor=STAGE_COLORS["Unresolved"], edgecolor="white", label="Unresolved"),
]
fig_platform_v2.legend(
    handles=platform_v2_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.012),
    ncol=4,
    frameon=False,
    fontsize=20,
    handlelength=1.8,
    columnspacing=2.2,
)
fig_platform_v2.subplots_adjust(left=0.055, right=0.99, top=0.965, bottom=0.085, hspace=0.58)

# Canonical combined V2 plus compatibility aliases for the three platform stems.
platform_v2_paths = []
for file_stem in [
    "cascade_platform_pcc_v2",
    "cascade_tmt_v2",
    "cascade_itraq_v2",
    "cascade_labelfree_v2",
]:
    platform_v2_paths.append(
        _save_pdf(
            fig_platform_v2,
            file_stem,
            "Pearson cascade flow — TMT, iTRAQ and Label-free",
        )
    )

COMBINED_PLATFORM_PCC_V2_PDF = platform_v2_paths[0]
print("Saved combined platform V2 cascade:", COMBINED_PLATFORM_PCC_V2_PDF)
plt.show()
plt.close(fig_platform_v2)


## 7. Final checks and provenance

The final audit verifies that Pearson and cosine have complete three-platform cascades, that **MSE appears only in the quantile analysis and never in a threshold cascade**, and that every exported table/figure is linked to the same frozen validation manifest.

In [ ]:
# Exact structural checks for Pearson/cosine cascades only.
expected_pairs = {(m, p) for m in CASCADE_SPECS for p in PLATFORM_ORDER}
observed_pairs = set(map(tuple, cascade_summary[["metric", "platform"]].drop_duplicates().to_numpy()))
if observed_pairs != expected_pairs:
    raise RuntimeError(f"Missing metric/platform cascade combinations: {sorted(expected_pairs - observed_pairs)}")
if cascade_summary["metric"].eq("mse").any() or cascade_assignments["metric"].eq("mse").any():
    raise RuntimeError("MSE must not enter a hard-threshold cascade in this notebook version.")
if set(cascade_summary["stage"].astype(str)) != set(STAGE_ORDER):
    raise RuntimeError("Cascade stages are incomplete.")
if set(best_proxy_composition["best_proxy_type"].astype(str)) - set(PROXY_ORDER):
    raise RuntimeError("Unexpected proxy formulation in best-proxy composition.")

# MSE quantile coverage must be complete for 3 models x 3 platforms.
expected_mse_pairs = {(p, m) for p in PLATFORM_ORDER for m in ["Best proxy", "ElasticNet", "XGBoost"]}
observed_mse_pairs = set(map(tuple, mse_quantile_summary[["platform", "model"]].to_numpy()))
if observed_mse_pairs != expected_mse_pairs:
    raise RuntimeError(f"Missing MSE quantile summaries: {sorted(expected_mse_pairs - observed_mse_pairs)}")
required_mse_cols = {f"q{int(round(100*q)):02d}" for q in MSE_QUANTILES}
if not required_mse_cols.issubset(mse_quantile_summary.columns):
    raise RuntimeError("MSE quantile output is incomplete.")

# Every analysis table is linked to the frozen notebook-7/8 result hash.
for frame_name, frame in {
    "proxy cohort metrics": proxy_cohort_metrics,
    "platform metric values": platform_metric_values,
    "best proxy selection": best_proxy_selection,
    "cascade assignments": cascade_assignments,
    "cascade summary": cascade_summary,
    "MSE quantiles": mse_quantile_summary,
}.items():
    pv.assert_manifest_column(frame, MANIFEST_HASH, frame_name)

# PDF-only figure registry. No PNG variables are created or required anywhere.
figure_paths = [Path(path) for path in figure_paths]
missing_figures = [str(path) for path in figure_paths if not path.exists()]
if missing_figures:
    raise FileNotFoundError(f"Expected PDF figures were not written: {missing_figures}")
if not figure_paths:
    raise RuntimeError("No PDF figures were registered.")
if any(path.suffix.lower() != ".pdf" for path in figure_paths):
    raise RuntimeError("Notebook 09 figure registry must contain PDF files only.")

figure_registry = pd.DataFrame([{
    "manifest_hash": MANIFEST_HASH,
    "figure": path.name,
    "path": str(path.resolve()),
    "sha256": pv.file_sha256(path),
} for path in figure_paths])
figure_registry_path = pv.write_table(
    figure_registry, LAYOUT["procedural"] / "best_proxy_procedural_figure_registry", index=False
)

outputs = {
    "proxy_cohort_metrics": str(Path(proxy_cohort_metrics_path).resolve()),
    "platform_metric_values": str(Path(platform_metric_values_path).resolve()),
    "best_proxy_selection": str(Path(best_proxy_selection_path).resolve()),
    "cascade_assignments": str(Path(cascade_assignments_path).resolve()),
    "cascade_summary": str(Path(cascade_summary_path).resolve()),
    "mse_quantile_summary": str(Path(mse_quantile_summary_path).resolve()),
    "best_proxy_composition": str(Path(best_proxy_composition_path).resolve()),
    "figure_registry": str(Path(figure_registry_path).resolve()),
    "current_contract_vs_frozen_linear_proxy_compatibility": str(Path(compatibility_path).resolve()),
}
provenance = pv.notebook_provenance(
    "09_procedural_best_proxy_paper_ready.ipynb",
    MANIFEST_HASH,
    {
        "manifest": str(MANIFEST_PATH.resolve()),
        "notebook8_primary_metrics": str((LAYOUT["tables"] / "external_primary_metrics_compact").resolve()),
        "ccle_reference": str(reference_path.resolve()),
        "linear_proxy_parameters": str(proxy_parameter_path.resolve()),
    },
    outputs,
)
provenance["manifest_resolution"] = {
    "current_manifest_hash": CURRENT_MANIFEST_HASH,
    "frozen_notebook7_notebook8_result_hash": MANIFEST_HASH,
    "hashes_match": bool(MANIFEST_MATCHES_CURRENT),
    "strict_manifest_match_requested": bool(STRICT_MANIFEST_MATCH),
    "notebook7_completion_hash_verified": bool(NOTEBOOK7_COMPLETION_VERIFIED),
    "current_contract_reproduces_frozen_linear_proxy": bool(CURRENT_CONTRACT_REPRODUCES_FROZEN_LINEAR_PROXY),
    "compatibility_rtol": COMPATIBILITY_RTOL,
    "compatibility_atol": COMPATIBILITY_ATOL,
}
provenance["procedural_design"] = {
    "proxy_variants": PROXY_ORDER,
    "proxy_selection": "best per protein x metric x platform using the external metric",
    "pearson_cosine_cascade_order": STAGE_ORDER,
    "pearson_thresholds": list(PEARSON_THRESHOLDS),
    "cosine_thresholds": list(COSINE_THRESHOLDS),
    "mse_analysis": "quantiles only; no hard MSE thresholds and no MSE success cascade",
    "mse_quantiles": list(MSE_QUANTILES),
    "plot_style": "PDF-only enlarged Notebook-4 stacked/flow cascades; Pearson platform cascades also combined in one shared-legend figure",
    "figure_directory": str(FIGURES_PDF.resolve()),
}
pv.atomic_json_dump(provenance, LAYOUT["procedural"] / "notebook09_best_proxy_provenance.json")

print("PASS — frozen notebook-7/8 result provenance resolved consistently.")
print("PASS — current contract reproduces the frozen LinearProxy metrics.")
print("Current manifest hash:", CURRENT_MANIFEST_HASH)
print("Frozen result hash:", MANIFEST_HASH)
print("Manifest hashes match:", MANIFEST_MATCHES_CURRENT)
print("PASS — Pearson TMT/iTRAQ/Label-free individual PDFs and one combined shared-legend PDF were generated.")
print("PASS — all Notebook 09 figures are PDF-only in figures_pdf; no PNG dependency remains.")
print("PASS — MSE uses quantiles only (q10/q25/q50/q75/q90); no hard MSE thresholds are present.")
print("Best-proxy selection remains an external oracle/upper-bound comparison and is separate from the primary linear-proxy analysis.")
display(figure_registry)
